In [9]:
import pandas as pd
import kagglehub
import os
from pathlib import Path

# 1. Definindo os caminhos das pastas (Arquitetura de Medalhão)
silver_path = Path("Data/Silver")
silver_path.mkdir(parents=True, exist_ok=True)

print("🛰️ Baixando dataset de jogadores do Kaggle...")
# 2. Download do dataset (Sempre pega a versão mais recente)
path_raw = kagglehub.dataset_download("hubertsidorowicz/football-players-stats-2025-2026")
csv_file = os.path.join(path_raw, "players_data_light-2025_2026.csv")

# 3. Lendo e Filtrando
print("🇮🇹 Filtrando jogadores da 'it Serie A'...")
df_raw = pd.read_csv(csv_file)
df_serie_a = df_raw[df_raw['Comp'] == 'it Serie A'].copy()

# 4. Criando a coluna de 'Date' (Evitando o erro do sort_values)
# Como o CSV original é um resumo, usamos o Rank (Rk) para simular o tempo
if 'Date' not in df_serie_a.columns:
    df_serie_a['Date'] = pd.to_datetime('2026-01-01') + pd.to_timedelta(df_serie_a['Rk'], unit='D')

# 5. Salvando na Camada Silver (Parquet é vida!)
output_file = silver_path / "serie_a_silver.parquet"
df_serie_a.to_parquet(output_file, index=False)

print(f"✨ Sucesso! O df_serie_a foi criado com {len(df_serie_a)} jogadores.")
print(f"📁 Salvo em: {output_file}")

# Exibindo os primeiros registros para conferir
df_serie_a.head()

🛰️ Baixando dataset de jogadores do Kaggle...
🇮🇹 Filtrando jogadores da 'it Serie A'...
✨ Sucesso! O df_serie_a foi criado com 578 jogadores.
📁 Salvo em: Data\Silver\serie_a_silver.parquet


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Date
12,13,Zakaria Aboukhlal,ma MAR,MF,Torino,it Serie A,26.0,2000.0,15,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-14
18,19,Francesco Acerbi,it ITA,DF,Inter,it Serie A,38.0,1988.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-20
22,23,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,26,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-24
26,27,Jayden Addai,nl NED,MF,Como,it Serie A,20.0,2005.0,12,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28
33,34,Vasilije Adžić,me MNE,MF,Juventus,it Serie A,19.0,2006.0,10,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-04


#2 Dados StatsBomb

In [24]:
# Carregar dados de eventos do StatsBomb (CSV ou JSON)
import glob
import json

path_sb = kagglehub.dataset_download("saurabhshahane/statsbomb-football-data")
file_sb = os.path.join(path_sb, "events.csv")

if os.path.exists(file_sb):
    df_sb = pd.read_csv(file_sb)
    print("\n" + "="*60)
    print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - CSV)")
    print("="*60)
    print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
    print("\n--- AMOSTRA DOS DADOS ---")
    print(df_sb.head())
else:
    # Fallback para o formato real desta base: JSONs em data/events
    event_json_files = glob.glob(os.path.join(path_sb, "**", "events", "*.json"), recursive=True)
    print(f"events.csv não encontrado. JSONs de eventos encontrados: {len(event_json_files)}")

    if not event_json_files:
        print(f"Nenhum arquivo de eventos foi encontrado em {path_sb}")
    else:
        records = []
        max_files = 20  # amostra para carregar rápido
        for fp in event_json_files[:max_files]:
            with open(fp, "r", encoding="utf-8") as f:
                events = json.load(f)
            if isinstance(events, list):
                records.extend(events)

        if records:
            df_sb = pd.json_normalize(records)
            print("\n" + "="*60)
            print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)")
            print("="*60)
            print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
            print("\nPrimeiras colunas:")
            print(df_sb.columns[::].tolist())
            print("\n--- AMOSTRA DOS DADOS ---")
            print(df_sb.head())
        else:
            print("JSONs encontrados, mas nenhum registro válido foi carregado.")

df_serie_a.head()

100%|██████████| 1.28G/1.28G [02:50<00:00, 8.03MB/s]

Extracting files...


events.csv não encontrado. JSONs de eventos encontrados: 3464

ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)
Linhas: 75,951 | Colunas: 137

Primeiras colunas:
['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type.id', 'type.name', 'possession_team.id', 'possession_team.name', 'play_pattern.id', 'play_pattern.name', 'team.id', 'team.name', 'tactics.formation', 'tactics.lineup', 'related_events', 'location', 'player.id', 'player.name', 'position.id', 'position.name', 'pass.recipient.id', 'pass.recipient.name', 'pass.length', 'pass.angle', 'pass.height.id', 'pass.height.name', 'pass.end_location', 'pass.body_part.id', 'pass.body_part.name', 'pass.type.id', 'pass.type.name', 'carry.end_location', 'pass.switch', 'pass.outcome.id', 'pass.outcome.name', 'ball_receipt.outcome.id', 'ball_receipt.outcome.name', 'under_pressure', 'duel.type.id', 'duel.type.name', 'pass.aerial_won', 'counterpress', 'interception.outcome.id', 'interception.outcome.name', 'off_

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Date
12,13,Zakaria Aboukhlal,ma MAR,MF,Torino,it Serie A,26.0,2000.0,15,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-14
18,19,Francesco Acerbi,it ITA,DF,Inter,it Serie A,38.0,1988.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-20
22,23,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,26,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-24
26,27,Jayden Addai,nl NED,MF,Como,it Serie A,20.0,2005.0,12,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28
33,34,Vasilije Adžić,me MNE,MF,Juventus,it Serie A,19.0,2006.0,10,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-04


#Parquet

In [27]:
import pandas as pd
import json
import os
import glob
from tqdm import tqdm # Barra de progresso

# 1. Configuração de Caminhos
# (Certifique-se que path_sb já foi definido na célula anterior)
event_json_files = glob.glob(os.path.join(path_sb, "**", "events", "*.json"), recursive=True)

print(f"📂 Iniciando o processamento de {len(event_json_files)} arquivos...")

# 2. Loop de Processamento Real
all_events = []

# Vamos processar em lotes para não estourar sua memória RAM
for fp in tqdm(event_json_files, desc="Convertendo JSON para Tabela"):
    try:
        with open(fp, "r", encoding="utf-8") as f:
            events = json.load(f)
        
        # Achatando o JSON (Flattening)
        df_temp = pd.json_normalize(events)
        
        # Guardando o ID da partida
        df_temp['match_id'] = os.path.basename(fp).replace(".json", "")
        
        all_events.append(df_temp)
    except Exception as e:
        continue # Pula arquivos corrompidos

# 3. Consolidação e Salvamento
if all_events:
    print("\n🔗 Concatenando tudo... (Isso pode demorar uns segundos)")
    df_final = pd.concat(all_events, ignore_index=True)
    
    # Removendo colunas totalmente vazias para economizar espaço
    df_final = df_final.dropna(axis=1, how='all')
    
    # SALVANDO EM PARQUET (Ouro!)
    output_parquet = "statsbomb_completo.parquet"
    df_final.to_parquet(output_parquet, compression="snappy", index=False)
    
    print(f"\n✅ SUCESSO TOTAL!")
    print(f"📊 Linhas processadas: {len(df_final):,}")
    print(f"💾 Arquivo gerado: {output_parquet}")
    
    # Agora sim, o head() para você ver a tabela
    display(df_final.head())
else:
    print("❌ Nenhum dado foi processado. Verifique os caminhos.")

📂 Iniciando o processamento de 3464 arquivos...


Convertendo JSON para Tabela: 100%|██████████| 3464/3464 [21:45<00:00,  2.65it/s]



🔗 Concatenando tudo... (Isso pode demorar uns segundos)

✅ SUCESSO TOTAL!
📊 Linhas processadas: 12,188,949
💾 Arquivo gerado: statsbomb_completo.parquet


,id,index,period,timestamp,minute,second,possession,duration,type.id,type.name,...,shot.redirect,goalkeeper.lost_out,player_off.permanent,goalkeeper.lost_in_play,goalkeeper.success_out,goalkeeper.success_in_play,goalkeeper.saved_to_post,half_end.early_video_end,shot.kick_off,goalkeeper.penalty_saved_to_post
0,9f6e2ecf-6685-45df-a62e-c2db3090f6c1,1,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0300039d-150d-41e4-b29a-76602ef002e6,2,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,491e8901-7630-4cc8-b57b-937dddff2eaa,3,1,00:00:00.000,0,0,1,0.000000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,757b85ad-ddfe-44d5-b893-c23a9fb709d8,4,1,00:00:00.000,0,0,1,0.000000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549567bd-36de-4ac8-b8dc-6b5d3f1e4be8,5,1,00:00:00.575,0,0,2,2.015669,30,Pass,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


visualização de dataset

In [ ]:
import polars as pl

print("📖 Carregando o Parquet com Polars...")

df = pl.read_parquet("statsbomb_completo.parquet")

# Normaliza nomes de colunas (substitui . por _)
df = df.rename({col: col.replace(".", "_") for col in df.columns})

print(f"✅ Carregado! Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

# ====================== COLUNAS IMPORTANTES ======================
print("\n🔍 Colunas disponíveis:")
print(df.columns)

# Colunas que realmente existem no seu dataset
col_type = "type_name"
col_player = "player_name"
col_team = "team_name"

print(f"\nUsando:")
print(f"   → Tipo de evento : {col_type}")
print(f"   → Jogador        : {col_player}")
print(f"   → Time           : {col_team}")

# ====================== 1. RESUMO GERAL ======================
print("\n📊 RESUMO GERAL DO DATASET:")

resumo = (
    df.group_by(col_type)
    .agg([
        pl.count().alias("Total_Eventos"),
        pl.col("match_id").n_unique().alias("Total_Partidas")
    ])
    .sort("Total_Eventos", descending=True)
    .head(10)
)

print(resumo)

# ====================== 2. ANÁLISE DO MSN ======================
msn_names = ["Lionel Andrés Messi Cuccittini", "Luis Alberto Suárez Díaz"]

df_msn = df.filter(pl.col(col_player).is_in(msn_names))

print(f"\n🎯 EVENTOS DO MSN:")
print(f"   Total eventos MSN : {df_msn.shape[0]:,}")
print(f"   Total eventos geral: {df.shape[0]:,}")
print(f"   Percentual        : {(df_msn.shape[0] / df.shape[0] * 100):.2f}%")

# Chutes do MSN
chutes_msn = df_msn.filter(pl.col(col_type) == "Shot")

print(f"   Total chutes do MSN: {chutes_msn.shape[0]:,}")
print(f"   Gols do MSN        : {chutes_msn.filter(pl.col('shot_outcome_name') == 'Goal').shape[0]:,}")

# ====================== 3. TOP 10 JOGADORES ======================
print("\n🔥 TOP 10 JOGADORES (mais eventos):")
top_players = (
    df.group_by(col_player)
    .agg(pl.count().alias("Total_Eventos"))
    .sort("Total_Eventos", descending=True)
    .head(10)
)
print(top_players)

# ====================== 4. TOP TIMES ======================
print("\n🏟️ TOP 5 TIMES (mais eventos):")
top_teams = (
    df.group_by(col_team)
    .agg(pl.count().alias("Total_Eventos"))
    .sort("Total_Eventos", descending=True)
    .head(5)
)
print(top_teams)

📖 Carregando o Parquet com Polars...
✅ Carregado! Shape: 12,188,949 linhas × 150 colunas

🔍 Colunas disponíveis:
['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type_id', 'type_name', 'possession_team_id', 'possession_team_name', 'play_pattern_id', 'play_pattern_name', 'team_id', 'team_name', 'tactics_formation', 'tactics_lineup', 'related_events', 'location', 'player_id', 'player_name', 'position_id', 'position_name', 'pass_recipient_id', 'pass_recipient_name', 'pass_length', 'pass_angle', 'pass_height_id', 'pass_height_name', 'pass_end_location', 'pass_body_part_id', 'pass_body_part_name', 'pass_type_id', 'pass_type_name', 'carry_end_location', 'pass_switch', 'pass_outcome_id', 'pass_outcome_name', 'ball_receipt_outcome_id', 'ball_receipt_outcome_name', 'under_pressure', 'duel_type_id', 'duel_type_name', 'pass_aerial_won', 'counterpress', 'interception_outcome_id', 'interception_outcome_name', 'off_camera', 'ball_recovery_recovery_failure', 'pass

C:\Users\kaiki\AppData\Local\Temp\ipykernel_15040\3287618990.py:32: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("Total_Eventos"),


shape: (10, 3)
┌───────────────┬───────────────┬────────────────┐
│ type_name     ┆ Total_Eventos ┆ Total_Partidas │
│ ---           ┆ ---           ┆ ---            │
│ str           ┆ u32           ┆ u32            │
╞═══════════════╪═══════════════╪════════════════╡
│ Pass          ┆ 3387760       ┆ 3464           │
│ Ball Receipt* ┆ 3167310       ┆ 3464           │
│ Carry         ┆ 2632570       ┆ 3464           │
│ Pressure      ┆ 1113859       ┆ 3464           │
│ Ball Recovery ┆ 366673        ┆ 3464           │
│ Duel          ┆ 257861        ┆ 3464           │
│ Clearance     ┆ 158993        ┆ 3464           │
│ Block         ┆ 132352        ┆ 3464           │
│ Dribble       ┆ 122047        ┆ 3464           │
│ Goal Keeper   ┆ 106574        ┆ 3464           │
└───────────────┴───────────────┴────────────────┘

🎯 EVENTOS DO MSN:
   Total eventos MSN : 158,827
   Total eventos geral: 12,188,949
   Percentual        : 1.30%
   Total chutes do MSN: 3,308
   Gols do MSN        : 6

C:\Users\kaiki\AppData\Local\Temp\ipykernel_15040\3287618990.py:61: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("Total_Eventos"))


In [ ]:
print(df_gold.columns)

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import brier_score_loss, roc_auc_score

# 1. CARREGAR O PARQUET (Muito mais rápido que JSON)
# df_raw = pd.read_parquet("statsbomb_completo.parquet")

# 2. GEOMETRIA VETORIZADA (Sem loops!)
def aplicar_geometria_bulk(df):
    x = df['location'].str[0].fillna(120)
    y = df['location'].str[1].fillna(40)
    
    # Distância Euclidiana
    df['distancia'] = np.sqrt((120 - x)**2 + (40 - y)**2)
    
    # Ângulo de Visão (Lei dos Cossenos Vetorizada)
    a = np.sqrt((120 - x)**2 + (36 - y)**2)
    b = np.sqrt((120 - x)**2 + (44 - y)**2)
    c = 8
    cos_theta = (a**2 + b**2 - c**2) / (2 * a * b)
    cos_theta = np.clip(cos_theta, -1, 1)
    df['angulo_visao'] = np.degrees(np.arccos(cos_theta))
    
    # Feature de Interação Proprietária
    # (Usamos o defensores_caminho extraído no pipeline do Parquet)
    df['dificuldade_geometrica'] = df['distancia'] * (1 + df.get('defensores_caminho', 0))
    return df

print("🧠 Calculando Features Matemáticas...")
df_gold = aplicar_geometria_bulk(df_gold)

# 3. TRATAMENTO DE CATEGORIAS
df_model = pd.get_dummies(df_gold, columns=['body_part'], drop_first=True)

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import brier_score_loss, roc_auc_score

# ====================== 1. PREPARAÇÃO FINAL DOS DADOS ======================
# Garantindo que temos apenas o que importa para o modelo
# Nota: 'df_model' deve ser o seu DataFrame vindo do Parquet/Dummies
features_base = [
    'distancia', 'angulo_visao', 'sob_pressao', 'defensores_caminho', 
    'gk_fora_posicao', 'pressao_construcao', 'dificuldade_geometrica', 'idade'
]
# Pega as colunas de body_part que o get_dummies criou
features_body = [col for col in df_model.columns if col.startswith('body_part_')]
features = features_base + features_body

X = df_model[features].fillna(0).astype(float)
y = df_model['is_goal'].astype(int)
groups = df_model['match_id'] # Essencial para o GroupKFold

# ====================== 2. SETUP DO CROSS-VALIDATION ======================
gkf = GroupKFold(n_splits=5)
brier_scores = []
auc_scores = []
models = []

print(f"🚀 Iniciando Treino de Elite: {len(X):,} chutes | 5 Partições de Jogos\n")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    # Peso dinâmico para equilibrar Gols vs Não-Gols
    pos_weight = y_tr.value_counts()[0] / y_tr.value_counts()[1]
    
    # O MODELO PARAMETRIZADO (Nível Bradesco/Cesar School)
    model = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=7,
        learning_rate=0.01,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_lambda=10,             # Regularização forte anti-overfitting
        scale_pos_weight=pos_weight,
        eval_metric='aucpr',       # Foca na precisão da classe rara (Gol)
        tree_method='hist',        # Otimização de performance para Big Data
        random_state=42 + fold
    )
    
    # Treino com Early Stopping (Evita desperdício de CPU)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        early_stopping_rounds=50,
        verbose=False
    )
    
    # Predições e Métricas
    probs = model.predict_proba(X_te)[:, 1]
    brier = brier_score_loss(y_te, probs)
    auc = roc_auc_score(y_te, probs)
    
    brier_scores.append(brier)
    auc_scores.append(auc)
    models.append(model)
    
    print(f"✅ Fold {fold+1} concluído | Brier: {brier:.4f} | AUC: {auc:.4f} | Jogos no Teste: {len(X_te)}")

# ====================== 3. VEREDITO FINAL ======================
print("\n" + "="*50)
print(f"🏆 RESULTADO CONSOLIDADO (AURA MSN v5.1)")
print(f"Média Brier Score: {np.mean(brier_scores):.4f} ± {np.std(brier_scores):.4f}")
print(f"Média ROC-AUC:     {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*50)

# Pegamos o melhor modelo (menor Brier) para o plot final
best_idx = np.argmin(brier_scores)
best_model = models[best_idx]

# Plot da importância das features para o Veredito
plt.figure(figsize=(12, 8))
xgb.plot_importance(best_model, importance_type='gain', max_num_features=12, color='skyblue')
plt.title(f"As Features que definem o Gol (Melhor Fold: {best_idx+1})")
plt.show()

🧠 Calculando Features Matemáticas...


NameError: name 'df_gold' is not defined

In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np

# 1. Configuração do Cross-Validation (Mantendo a separação por partida)
gkf = GroupKFold(n_splits=5)
brier_scores = []
auc_scores = []

print(f"🚀 Iniciando Treino Pesado: {len(X)} instâncias em 5 Folds...\n")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    # Cálculo dinâmico do peso para cada fold
    pos_weight = y_tr.value_counts()[0] / y_tr.value_counts()[1]
    
    # O SEU MODELO PARAMETRIZADO
    model = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=7,
        learning_rate=0.01,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_lambda=10,
        scale_pos_weight=pos_weight,
        eval_metric='aucpr', # Foca na precisão dos gols (classe rara)
        tree_method='hist',  # OTIMIZAÇÃO PARA BIG DATA
        random_state=42 + fold
    )
    
    # Treinamento com Early Stopping
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        early_stopping_rounds=50, # Para se não melhorar em 50 árvores
        verbose=False
    )
    
    # Avaliação
    probs = model.predict_proba(X_te)[:, 1]
    brier = brier_score_loss(y_te, probs)
    auc = roc_auc_score(y_te, probs)
    
    brier_scores.append(brier)
    auc_scores.append(auc)
    
    print(f"✅ Fold {fold+1} concluído | Brier: {brier:.4f} | AUC: {auc:.4f}")

# ====================== RESULTADO FINAL ======================
print("\n" + "="*40)
print(f"🏆 VEREDITO FINAL (DATASET GIGANTE)")
print(f"Média Brier Score: {np.mean(brier_scores):.4f} ± {np.std(brier_scores):.4f}")
print(f"Média ROC-AUC:     {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*40)

##teste

In [ ]:
import json
import pandas as pd
import os
from pathlib import Path

# 1. Localizar o arquivo de Competitions (O Catálogo)
path_competitions = os.path.join(path_sb, "data", "competitions.json")

with open(path_competitions, "r", encoding="utf-8") as f:
    comps = json.load(f)

df_comps = pd.json_normalize(comps)

# 2. Filtrar pela temporada mais recente (ex: 2020/2021 ou a maior que tiver)
# Ordenamos por season_id para pegar o que há de mais novo no dataset
df_recent_comps = df_comps.sort_values(by="season_id", ascending=False)

print("🏆 Temporadas mais recentes encontradas:")
print(df_recent_comps[['competition_name', 'season_name', 'competition_id', 'season_id']].head(5))

# 3. Pegar os IDs da temporada mais recente disponível (ex: La Liga 2020/2021)
target_comp = df_recent_comps.iloc[0]['competition_id']
target_season = df_recent_comps.iloc[0]['season_id']

# 4. Localizar os Matches dessa temporada específica
path_matches = os.path.join(path_sb, "data", "matches", str(target_comp), f"{target_season}.json")

with open(path_matches, "r", encoding="utf-8") as f:
    matches = json.load(f)

# Pegamos os IDs de todos os jogos dessa temporada "Elite"
match_ids = [m['match_id'] for m in matches]
print(f"✅ Encontrados {len(match_ids)} jogos recentes da temporada {df_recent_comps.iloc[0]['season_name']}.")

🏆 Temporadas mais recentes encontradas:
     competition_name season_name  competition_id  season_id
71  UEFA Women's Euro        2025              53        315
21       Copa America        2024             223        282
68          UEFA Euro        2024              55        282
0       1. Bundesliga   2023/2024               9        281
24       Copa del Rey   1977/1978              87        279
✅ Encontrados 31 jogos recentes da temporada 2025.
